# Phase 1 · Taxonomy acquisition and statistics

This phase acquires the complete taxonomy for **bacteria, fungi, and viruses**. The three inputs follow different source workflows: bacterial taxonomy and aliases are migrated from the local V2 LPSN-derived release; fungal names come from the pinned Species Fungorum export; viral names come from the pinned ICTV Master Species List, with isolate and strain metadata from the ICTV Virus Metadata Resource (VMR).

The cells below show the source files and releases, inspect the source-specific adapters, run the acquisition, and report normalized output statistics for every domain. Phase 1 is independent of the selected group setting for Phases 2–5.


In [ ]:
from pathlib import Path
import inspect
import os
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.taxonomy.adapters import (
    ADAPTERS,
    load_ictv_vmr_export,
    read_source_file,
)
from pathogen_atlas_v3.migration.version2 import migrate_v2_snapshot
from pathogen_atlas_v3.pipelines.phase1_taxonomy import run_phase1

CONFIG_PATH = PROJECT_ROOT / "configs/taxonomy/source_files.yaml"
DEFAULT_CONFIG_PATH = PROJECT_ROOT / "configs/default.yaml"
SETTINGS = load_yaml(CONFIG_PATH)
DEFAULT_CONFIG = load_yaml(DEFAULT_CONFIG_PATH)
TAXONOMY_ROOT = PROJECT_ROOT / DEFAULT_CONFIG["paths"]["taxonomy_release"]
MIGRATION_ROOT = PROJECT_ROOT / DEFAULT_CONFIG["paths"]["migration_source"]
print("Project:", PROJECT_ROOT)
print("Phase 1 covers all domains; Phase 2–5 selection:", DEFAULT_CONFIG["run"]["organism_groups"])
print("Normalized taxonomy output:", TAXONOMY_ROOT)


## Acquisition inputs · all three domains

In [ ]:
# Inspect the pinned source files and raw row counts before running the adapters.
source_rows = []

# Bacteria: V2 local migration input. It includes canonical species and synonym releases.
for relative in (
    "01_species/release/species_canonical.parquet",
    "01_species/release/species_taxonomy.parquet",
    "01_species/release/species_synonym_mapping.parquet",
):
    path = MIGRATION_ROOT / relative
    row_count = len(pd.read_parquet(path)) if path.exists() else None
    source_rows.append({
        "domain": "bacteria", "acquisition": "V2 migration · LPSN-derived records",
        "release": "migrated-v2-lpsn-local-capture", "input": str(path),
        "exists": path.exists(), "input rows": row_count,
    })

# Fungi: accepted species and synonym mappings are derived from Species Fungorum.
fungi = SETTINGS["sources"]["fungi"]
fungi_path = PROJECT_ROOT / fungi["path"]
try:
    fungi_raw_count = len(read_source_file(fungi_path)) if fungi_path.exists() else None
except Exception as exc:
    fungi_raw_count = f"read error: {type(exc).__name__}: {exc}"
source_rows.append({
    "domain": "fungi", "acquisition": fungi["adapter"], "release": fungi["release"],
    "input": str(fungi_path), "exists": fungi_path.exists(), "input rows": fungi_raw_count,
})

# Viruses: the ICTV MSL supplies species taxonomy; its VMR supplies isolate-level metadata.
viruses = SETTINGS["sources"]["viruses"]
for role, path_key, release_key, sheet in (
    ("species taxonomy", "path", "release", "MSL"),
    ("variant / isolate metadata", "variant_metadata_path", "variant_metadata_release", "VMR MSL41"),
):
    path = PROJECT_ROOT / viruses[path_key]
    try:
        raw_count = len(read_source_file(path, sheet_name=sheet)) if path.exists() else None
    except Exception as exc:
        raw_count = f"read error: {type(exc).__name__}: {exc}"
    adapter = viruses.get("variant_metadata_adapter", "ictv_msl") if role.startswith("variant") else viruses["adapter"]
    source_rows.append({
        "domain": "viruses", "acquisition": f"{adapter} · {role}",
        "release": viruses[release_key], "input": str(path),
        "exists": path.exists(), "input rows": raw_count,
    })

SOURCE_INVENTORY = pd.DataFrame(source_rows)
display(SOURCE_INVENTORY)
if not SOURCE_INVENTORY["exists"].all():
    display(SOURCE_INVENTORY.loc[~SOURCE_INVENTORY["exists"]])


## Acquisition code · source-specific methods

In [ ]:
# Bacteria are migrated from the local V2 tables; fungi and viruses use their own taxonomy adapters.
for label, function in (
    ("Bacteria · V2 migration", migrate_v2_snapshot),
    ("Fungi · Species Fungorum adapter", ADAPTERS["fungi"]),
    ("Viruses · ICTV MSL adapter", ADAPTERS["viruses"]),
    ("Viruses · ICTV VMR isolate/alias adapter", load_ictv_vmr_export),
):
    print(f"\n### {label} · {function.__module__}.{function.__name__}\n")
    print(inspect.getsource(function))

print("\n### Phase 1 pipeline entry point\n")
print(inspect.getsource(run_phase1))


## Run acquisition and rebuild normalized taxonomy

In [ ]:
RUN_PHASE = True
if RUN_PHASE:
    TAXONOMY_COUNTS = run_phase1(
        CONFIG_PATH,
        migration_source=MIGRATION_ROOT,
        output_root=PROJECT_ROOT,
    )
    display(pd.DataFrame([TAXONOMY_COUNTS]).T.rename(columns={0: "records / count"}))
else:
    TAXONOMY_COUNTS = {"status": "not_run", "reason": "Set RUN_PHASE=True to rebuild taxonomy tables."}
    display(TAXONOMY_COUNTS)


## Normalized output statistics · bacteria, fungi, and viruses

In [ ]:
# These are counts from the normalized Phase 1 release, not raw source row counts.
SPECIES_PATH = TAXONOMY_ROOT / "species.parquet"
ALIASES_PATH = TAXONOMY_ROOT / "aliases.parquet"
VARIANTS_PATH = TAXONOMY_ROOT / "virus_variants.parquet"

if not SPECIES_PATH.exists() or not ALIASES_PATH.exists():
    raise FileNotFoundError("Run Phase 1 first; normalized species or aliases release is missing.")

species = pd.read_parquet(SPECIES_PATH)
aliases = pd.read_parquet(ALIASES_PATH)
variants = pd.read_parquet(VARIANTS_PATH) if VARIANTS_PATH.exists() else pd.DataFrame()
summary_rows = []
for domain in ("bacteria", "fungi", "viruses"):
    domain_species = species.loc[species["organism_group"].astype(str).eq(domain)]
    domain_aliases = aliases.loc[aliases["organism_group"].astype(str).eq(domain)]
    domain_variants = variants.loc[variants.get("organism_group", pd.Series(dtype=str)).astype(str).eq(domain)] if not variants.empty else pd.DataFrame()
    summary_rows.append({
        "domain": domain,
        "species records": len(domain_species),
        "accepted-name rows": int(domain_aliases["name_type"].astype(str).eq("accepted").sum()) if "name_type" in domain_aliases else 0,
        "synonym / strain-alias rows": int(domain_aliases["name_type"].astype(str).ne("accepted").sum()) if "name_type" in domain_aliases else 0,
        "unresolved alias mappings": int(domain_aliases["organism_id"].isna().sum()) if "organism_id" in domain_aliases else 0,
        "ambiguous species records": int(domain_species["mapping_status"].astype(str).str.contains("ambiguous", case=False, na=False).sum()) if "mapping_status" in domain_species else 0,
        "source releases": "; ".join(sorted(domain_species["taxonomy_release"].dropna().astype(str).unique())) if "taxonomy_release" in domain_species else "",
        "virus variant / isolate records": len(domain_variants),
        "unresolved virus variants": int(domain_variants["organism_id"].isna().sum()) if "organism_id" in domain_variants else 0,
    })

DOMAIN_STATISTICS = pd.DataFrame(summary_rows).set_index("domain")
display(DOMAIN_STATISTICS)
print(f"\nCombined taxonomy: {len(species):,} species records; {len(aliases):,} alias rows.")
print("Species output:", SPECIES_PATH)
print("Aliases output:", ALIASES_PATH)
print("Virus variants output:", VARIANTS_PATH)
